In [41]:
import pandas as pd
import numpy as np

In [42]:
parking = pd.read_csv("../data/원본/경기도 화성시_주정차단속현황_20250430.csv", encoding="cp949")
pop = pd.read_csv("../data/원본/유동인구_시군구_시간대별_집계.csv", encoding="cp949")
shop = pd.read_csv("../data/원본/소상공인시장진흥공단_상가(상권)정보_경기_202603.csv", encoding="utf-8")

In [43]:
# 주정차 단속 데이터 컬럼 확인
parking.columns

Index(['단속기관', '과태료 부과연월', '과태료 부과건수', '부과금액(천원)'], dtype='object')

In [44]:
parking = parking.rename(columns={
    "과태료 부과연월": "기준년월",
    "과태료 부과건수": "단속건수",
    "부과금액(천원)": "부과금액_천원"
})

parking["기준년월"] = pd.to_datetime(parking["기준년월"], errors="coerce")

parking["연도"] = parking["기준년월"].dt.year
parking["월"] = parking["기준년월"].dt.month  # 연도, 월 추출

parking["단속기관"] = parking["단속기관"].astype(str) # 문자열 처리

parking = parking.dropna(subset=["기준년월", "단속건수"]) # 결측치 제거

parking_monthly = parking.groupby(["연도", "월"], as_index=False).agg({
    "단속건수": "sum",
    "부과금액_천원": "sum"
}) # 월별 집계

In [45]:
# 유동인구 데이터 컬럼 확인
pop.columns

Index(['기준년월', '시군구코드', '시간대코드', '유동인구수', '유동인구수비율', '전월대비증감값', '전월대비증감비율',
       '전년동월대비증감값', '전년동월대비증감비율'],
      dtype='object')

In [46]:
pop = pop.rename(columns={
    "시간대코드": "시간대",
    "유동인구수": "유동인구수"
})

pop["기준년월"] = pd.to_datetime(pop["기준년월"], errors="coerce")

pop["연도"] = pop["기준년월"].dt.year
pop["월"] = pop["기준년월"].dt.month


pop_hs = pop[pop["시군구코드"] == 41590].copy() # 화성시 시군구코드: 41590

pop_hs = pop_hs[pop_hs["시간대"] == "TOT"].copy()

pop_hs = pop_hs.dropna(subset=["기준년월", "시간대", "유동인구수"])

pop_monthly = pop_hs.groupby(["연도", "월"], as_index=False).agg({"유동인구수": "sum"})

In [47]:
# 상권 데이터 컬럼 확인
shop.columns

Index(['상가업소번호', '상호명', '지점명', '상권업종대분류코드', '상권업종대분류명', '상권업종중분류코드',
       '상권업종중분류명', '상권업종소분류코드', '상권업종소분류명', '표준산업분류코드', '표준산업분류명', '시도코드',
       '시도명', '시군구코드', '시군구명', '행정동코드', '행정동명', '법정동코드', '법정동명', '지번코드',
       '대지구분코드', '대지구분명', '지번본번지', '지번부번지', '지번주소', '도로명코드', '도로명', '건물본번지',
       '건물부번지', '건물관리번호', '건물명', '도로명주소', '구우편번호', '신우편번호', '동정보', '층정보',
       '호정보', '경도', '위도'],
      dtype='object')

In [48]:
shop_hs = shop[shop["시군구명"].astype(str).str.contains("화성시")].copy() 

shop_hs = shop_hs[
    [
        "상호명",
        "상권업종대분류명",
        "상권업종중분류명",
        "상권업종소분류명",
        "시군구명",
        "행정동명",
        "법정동명",
        "지번주소",
        "도로명주소",
        "경도",
        "위도"
    ]
]

shop_hs = shop_hs.dropna(subset=["경도", "위도"])
shop_hs = shop_hs.drop_duplicates() 

# 업종별 상가 수
shop_category = shop_hs.groupby("상권업종대분류명", as_index=False).agg({"상호명": "count"}).rename(columns={"상호명": "상가수"})

# 행정동별 상가 수
shop_dong = shop_hs.groupby("행정동명", as_index=False).agg({
    "상호명": "count"
}).rename(columns={"상호명": "상가수"})

# 전체 상가 수
total_shop_count = len(shop_hs)

In [49]:
# 월별 통합 데이터 생성
final_monthly = pd.merge(
    parking_monthly,
    pop_monthly,
    on=["연도", "월"],
    how="left"
)

final_monthly["총상가수"] = total_shop_count

final_monthly["단속건수"] = final_monthly["단속건수"].fillna(0)
final_monthly["유동인구수"] = final_monthly["유동인구수"].fillna(0)

In [50]:
# 지역별 위험도를 정량적으로 비교할 수 있도록 파생변수 생성
final_monthly["건당평균부과금액_천원"] = (final_monthly["부과금액_천원"] / final_monthly["단속건수"]).replace([np.inf, -np.inf], 0).fillna(0)

final_monthly["유동인구_대비_단속비율"] = (final_monthly["단속건수"] / final_monthly["유동인구수"]).replace([np.inf, -np.inf], 0).fillna(0)

In [51]:
# 저장
final_monthly.to_csv("../data/전처리_기존/월별통합데이터_전처리.csv", index=False, encoding="utf-8-sig")
shop_hs.to_csv("../data/전처리_기존/화성시상권데이터_전처리.csv", index=False, encoding="utf-8-sig")
shop_category.to_csv("../data/전처리_기존/업종별상권집계_전처리.csv", index=False, encoding="utf-8-sig")
shop_dong.to_csv("../data/전처리_기존/행정동별상권집계_전처리.csv", index=False, encoding="utf-8-sig")

In [52]:
print("월별 통합 데이터")
display(final_monthly.head())

print("화성시 상권 데이터")
display(shop_hs.head())

print("업종별 상권 집계")
display(shop_category.head())

print("행정동별 상권 집계")
display(shop_dong.head())

월별 통합 데이터


,연도,월,단속건수,부과금액_천원,유동인구수,총상가수,건당평균부과금액_천원,유동인구_대비_단속비율
0,2021,1,13830,556613,0.0,52635,40.246782,0.0
1,2021,2,16421,658155,0.0,52635,40.080080,0.0
2,2021,3,21809,887194,0.0,52635,40.680178,0.0
3,2021,4,19742,798416,0.0,52635,40.442508,0.0
4,2021,5,19438,803594,0.0,52635,41.341393,0.0


화성시 상권 데이터


,상호명,상권업종대분류명,상권업종중분류명,상권업종소분류명,시군구명,행정동명,법정동명,지번주소,도로명주소,경도,위도
66,디앤아트,과학·기술,기술 서비스,기타 엔지니어링 서비스업,화성시 동탄구,동탄3동,능동,경기도 화성시 동탄구 능동 1064-5,경기도 화성시 동탄구 동탄원천로 354-28,127.058722,37.218269
69,에이치알씨앤씨,과학·기술,본사·경영 컨설팅,경영 컨설팅업,화성시 동탄구,동탄5동,영천동,경기도 화성시 동탄구 영천동 846-1,경기도 화성시 동탄구 동탄대로 677-12,127.100157,37.214934
70,사이로건축사사무소,과학·기술,기술 서비스,건축 설계 및 관련 서비스업,화성시 동탄구,동탄5동,영천동,경기도 화성시 동탄구 영천동 823-6,경기도 화성시 동탄구 동탄첨단산업1로 27,127.089472,37.211023
82,엘이엔티,과학·기술,기술 서비스,기타 엔지니어링 서비스업,화성시 만세구,향남읍,향남읍,경기도 화성시 만세구 향남읍 동오리 224-3,경기도 화성시 만세구 향남읍 발안로464번길 14-8,126.957157,37.129711
90,낼에프에이,과학·기술,기술 서비스,기타 엔지니어링 서비스업,화성시 병점구,화산동,안녕동,경기도 화성시 병점구 안녕동 176-172,경기도 화성시 병점구 안녕남로50번길 16,126.987531,37.196216


업종별 상권 집계


,상권업종대분류명,상가수
0,과학·기술,5073
1,교육,5404
2,보건의료,1315
3,부동산,2639
4,소매,12429


행정동별 상권 집계


,행정동명,상가수
0,기배동,548
1,남양읍,3507
2,동탄1동,4709
3,동탄2동,1074
4,동탄3동,1355


---
## 유동인구 재전처리 (증감비율 방식)

**문제:** 유동인구수 원시 데이터의 측정 방식이 2022년부터 일부 달에서 바뀌어,
절대값 비교가 불가능합니다. (2021년 ~40만 vs 2022년 ~500만)

**해결:** 절대값(유동인구수) 대신 을 사용합니다.
이 값은 "작년 같은 달 대비 몇 % 변화"를 의미하므로 단위 변경에 영향받지 않습니다.

In [53]:
# 유동인구 원시 데이터 재로드
pop_raw = pd.read_csv("../data/원본/유동인구_시군구_시간대별_집계.csv", encoding="cp949")

# 화성시(41590) + TOT 시간대만 필터
pop_hs2 = pop_raw[
    (pop_raw["시군구코드"] == 41590) &
    (pop_raw["시간대코드"] == "TOT")
].copy()

# 기준년월 파싱
pop_hs2["기준년월"] = pd.to_datetime(pop_hs2["기준년월"].astype(str), format="%Y%m", errors="coerce")
pop_hs2["연도"] = pop_hs2["기준년월"].dt.year
pop_hs2["월"]   = pop_hs2["기준년월"].dt.month

# 필요한 컬럼만 선택
pop_hs2 = pop_hs2[["연도", "월", "유동인구수", "전년동월대비증감비율"]].dropna(subset=["연도", "월"])

print(pop_hs2.head(10))
print("Shape:", pop_hs2.shape)

        연도   월      유동인구수  전년동월대비증감비율
107   2020  10  371633.34   14.452191
762   2018   1  305632.44    0.000000
1024  2020  11  377527.72    2.141715
1486  2020  12  371672.84    2.008700
1920  2018   2  294914.84    0.000000
2382  2018   3  322200.53    0.000000
3146  2018   4  318360.78    0.000000
3608  2018   5  317796.28    0.000000
4070  2018   6  322913.90    0.000000
4223  2018   7  330273.20    0.000000
Shape: (90, 4)


In [54]:
# 단속 데이터 재로드 및 월별 집계
parking2 = pd.read_csv("../data/원본/경기도 화성시_주정차단속현황_20250430.csv", encoding="cp949")
parking2 = parking2.rename(columns={
    "과태료 부과연월": "기준년월",
    "과태료 부과건수": "단속건수",
    "부과금액(천원)": "부과금액_천원"
})
parking2["기준년월"] = pd.to_datetime(parking2["기준년월"], errors="coerce")
parking2["연도"] = parking2["기준년월"].dt.year
parking2["월"]   = parking2["기준년월"].dt.month
parking2 = parking2.dropna(subset=["기준년월", "단속건수"])

parking_monthly2 = parking2.groupby(["연도", "월"], as_index=False).agg({
    "단속건수": "sum",
    "부과금액_천원": "sum"
})

In [55]:
# 월별 통합 데이터 생성 (전년동월대비증감비율 사용)
final_monthly_v2 = pd.merge(
    parking_monthly2,
    pop_hs2[['연도', '월', '전년동월대비증감비율']],
    on=['연도', '월'],
    how='left'
)

final_monthly_v2['총상가수'] = total_shop_count

# 파생변수 1: 단속 1건당 평균 부과금액
final_monthly_v2['건당평균부과금액_천원'] = (
    final_monthly_v2['부과금액_천원'] / final_monthly_v2['단속건수']
).replace([np.inf, -np.inf], 0).fillna(0)

# 파생변수 2: 단속건수 전월 대비 증감률
final_monthly_v2 = final_monthly_v2.sort_values(['연도', '월']).reset_index(drop=True)
final_monthly_v2['단속건수_전월대비증감률'] = (
    final_monthly_v2['단속건수'].pct_change() * 100
).round(2)

# 파생변수 3: 계절 컬럼 (모델 피처용)
def month_to_season(m):
    if m in [3, 4, 5]:     return '봄'
    elif m in [6, 7, 8]:   return '여름'
    elif m in [9, 10, 11]: return '가을'
    else:                  return '겨울'

final_monthly_v2['계절'] = final_monthly_v2['월'].apply(month_to_season)

display(final_monthly_v2.head(15))
print('Shape:', final_monthly_v2.shape)
print('컬럼:', final_monthly_v2.columns.tolist())

,연도,월,단속건수,부과금액_천원,전년동월대비증감비율,총상가수,건당평균부과금액_천원,단속건수_전월대비증감률,계절
0,2021,1,13830,556613,6.229943,52635,40.246782,NaN,겨울
1,2021,2,16421,658155,7.566825,52635,40.080080,18.73,겨울
2,2021,3,21809,887194,14.645880,52635,40.680178,32.81,봄
3,2021,4,19742,798416,13.404402,52635,40.442508,-9.48,봄
4,2021,5,19438,803594,14.262414,52635,41.341393,-1.54,봄
5,2021,6,20363,853352,15.147522,52635,41.906988,4.76,여름
6,2021,7,18921,798427,14.581026,52635,42.197928,-7.08,여름
7,2021,8,18628,789221,19.065617,52635,42.367458,-1.55,여름
8,2021,9,18341,779200,15.490251,52635,42.484052,-1.54,가을
9,2021,10,19192,819164,17.896994,52635,42.682576,4.64,가을


Shape: (52, 9)
컬럼: ['연도', '월', '단속건수', '부과금액_천원', '전년동월대비증감비율', '총상가수', '건당평균부과금액_천원', '단속건수_전월대비증감률', '계절']


In [56]:
# NaN 및 극단값 처리

# 1) 단속건수_전월대비증감률: 시계열 첫 행(2021.01) NaN -> 해당 행 제거
final_monthly_v2 = final_monthly_v2.dropna(subset=['단속건수_전월대비증감률']).reset_index(drop=True)

# 2) 전년동월대비증감비율: 유동인구 측정방식 변경 구간에서 1000%+ 극단값 발생
#    -100 ~ +100% 범위를 벗어나는 값은 0(중립)으로 대체
abnormal_mask = final_monthly_v2['전년동월대비증감비율'].abs() > 100
final_monthly_v2.loc[abnormal_mask, '전년동월대비증감비율'] = 0

print('처리 완료')
print('Shape:', final_monthly_v2.shape)
print('극단값 처리된 행 수:', abnormal_mask.sum())
print('NaN 잔여:', final_monthly_v2.isnull().sum().to_dict())
display(final_monthly_v2[['연도', '월', '전년동월대비증감비율']].head(20))

처리 완료
Shape: (51, 9)
극단값 처리된 행 수: 11
NaN 잔여: {'연도': 0, '월': 0, '단속건수': 0, '부과금액_천원': 0, '전년동월대비증감비율': 0, '총상가수': 0, '건당평균부과금액_천원': 0, '단속건수_전월대비증감률': 0, '계절': 0}


,연도,월,전년동월대비증감비율
0,2021,2,7.566825
1,2021,3,14.645880
2,2021,4,13.404402
3,2021,5,14.262414
4,2021,6,15.147522
5,2021,7,14.581026
6,2021,8,19.065617
7,2021,9,15.490251
8,2021,10,17.896994
9,2021,11,16.742535


In [57]:
# 저장
final_monthly_v2.to_csv("../data/전처리_기존/월별통합데이터_전처리_v2.csv", index=False, encoding="utf-8-sig")
print("저장 완료: 월별통합데이터_전처리_v2.csv")
print(final_monthly_v2.dtypes)

저장 완료: 월별통합데이터_전처리_v2.csv
연도                int32
월                 int32
단속건수              int64
부과금액_천원           int64
전년동월대비증감비율      float64
총상가수              int64
건당평균부과금액_천원     float64
단속건수_전월대비증감률    float64
계절               object
dtype: object


---
## 공간 분석 — 행정동별 위험도 점수 계산

**목표:** 화성시 29개 행정동의 불법 주정차 위험도를 정량화

**방법:**
1. DBSCAN — 상권 위경도 기반 밀집 클러스터 탐지
2. 행정동별 위험도 점수 = 상가 밀도 + 음식/소매 비율 (주차 수요 지표)

In [58]:
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

# 상권 데이터 로드
shop_df = pd.read_csv(
    '../data/전처리_기존/화성시상권데이터_전처리.csv',
    encoding='utf-8-sig'
)

print('상권 데이터 Shape:', shop_df.shape)
print('행정동 수:', shop_df['행정동명'].nunique())
display(shop_df.head(3))

상권 데이터 Shape: (52635, 11)
행정동 수: 29


,상호명,상권업종대분류명,상권업종중분류명,상권업종소분류명,시군구명,행정동명,법정동명,지번주소,도로명주소,경도,위도
0,디앤아트,과학·기술,기술 서비스,기타 엔지니어링 서비스업,화성시 동탄구,동탄3동,능동,경기도 화성시 동탄구 능동 1064-5,경기도 화성시 동탄구 동탄원천로 354-28,127.058722,37.218269
1,에이치알씨앤씨,과학·기술,본사·경영 컨설팅,경영 컨설팅업,화성시 동탄구,동탄5동,영천동,경기도 화성시 동탄구 영천동 846-1,경기도 화성시 동탄구 동탄대로 677-12,127.100157,37.214934
2,사이로건축사사무소,과학·기술,기술 서비스,건축 설계 및 관련 서비스업,화성시 동탄구,동탄5동,영천동,경기도 화성시 동탄구 영천동 823-6,경기도 화성시 동탄구 동탄첨단산업1로 27,127.089472,37.211023


In [59]:
# DBSCAN 클러스터링
# haversine metric eps는 라디안 단위: 500m = 0.5/6371 = 0.0000785rad
# eps=0.00008 ≈ 반경 509m, min_samples=15: 500m 안에 15개 이상 상가면 클러스터
coords = shop_df[['위도', '경도']].values

db = DBSCAN(eps=0.00008, min_samples=15, algorithm='ball_tree', metric='haversine')
shop_df['cluster'] = db.fit_predict(np.radians(coords))

n_clusters = len(set(shop_df['cluster'])) - (1 if -1 in shop_df['cluster'].values else 0)
n_noise    = (shop_df['cluster'] == -1).sum()

print(f'클러스터 수: {n_clusters}개')
print(f'노이즈(고립 상가): {n_noise}개 ({n_noise/len(shop_df)*100:.1f}%)')
print()

# 행정동별 클러스터 수 집계
cluster_per_dong = (
    shop_df[shop_df['cluster'] != -1]
    .groupby('행정동명')['cluster']
    .nunique()
    .reset_index()
    .rename(columns={'cluster': '클러스터수'})
)
print('=== 행정동별 상권 클러스터 수 (상위 10) ===')
display(cluster_per_dong.sort_values('클러스터수', ascending=False).head(10))

클러스터 수: 42개
노이즈(고립 상가): 1014개 (1.9%)

=== 행정동별 상권 클러스터 수 (상위 10) ===


,행정동명,클러스터수
22,우정읍,10
19,서신면,7
20,송산면,6
23,장안면,5
21,양감면,5
12,매송면,5
26,팔탄면,4
16,봉담읍,3
11,마도면,3
1,남양읍,3


In [60]:
# 행정동별 위험도 점수 계산

# 1) 행정동별 총 상가수
dong_total = (
    shop_df.groupby('행정동명')
    .size()
    .reset_index(name='총상가수')
)

# 2) 행정동별 음식+소매 상가수 (주차 수요 높은 업종)
high_demand = shop_df[shop_df['상권업종대분류명'].isin(['음식', '소매'])]
dong_demand = (
    high_demand.groupby('행정동명')
    .size()
    .reset_index(name='주차수요상가수')
)

# 3) 행정동별 클러스터 수 (상권 밀집도)
risk_df = dong_total.merge(dong_demand, on='행정동명', how='left')
risk_df = risk_df.merge(cluster_per_dong, on='행정동명', how='left')
risk_df['클러스터수'] = risk_df['클러스터수'].fillna(0)
risk_df['주차수요상가수'] = risk_df['주차수요상가수'].fillna(0)

# 4) 음식+소매 비율
risk_df['주차수요비율'] = risk_df['주차수요상가수'] / risk_df['총상가수']

# 5) 정규화 후 위험도 점수 (0~10점)
scaler = MinMaxScaler(feature_range=(0, 10))
risk_df['상가밀도점수']  = scaler.fit_transform(risk_df[['총상가수']]).round(2)
risk_df['수요비율점수']  = scaler.fit_transform(risk_df[['주차수요비율']]).round(2)
risk_df['클러스터점수'] = scaler.fit_transform(risk_df[['클러스터수']]).round(2)

# 최종 위험도 = 상가밀도(40%) + 수요비율(30%) + 클러스터(30%)
risk_df['위험도점수'] = (
    risk_df['상가밀도점수']  * 0.4 +
    risk_df['수요비율점수']  * 0.3 +
    risk_df['클러스터점수'] * 0.3
).round(2)

# 위험 등급 분류
risk_df['위험등급'] = pd.cut(
    risk_df['위험도점수'],
    bins=[0, 3, 6, 10],
    labels=['저위험', '중위험', '고위험'],
    include_lowest=True
)

risk_df = risk_df.sort_values('위험도점수', ascending=False).reset_index(drop=True)

print('=== 행정동별 위험도 점수 (전체) ===')
display(risk_df[['행정동명','총상가수','주차수요비율','클러스터수','위험도점수','위험등급']])

=== 행정동별 위험도 점수 (전체) ===


,행정동명,총상가수,주차수요비율,클러스터수,위험도점수,위험등급
0,우정읍,1372,0.628280,10,6.16,고위험
1,향남읍,5389,0.574132,2,6.15,고위험
2,서신면,1188,0.690236,7,5.65,중위험
3,봉담읍,4481,0.549431,3,5.50,중위험
4,남양읍,3507,0.549187,3,4.72,중위험
5,송산면,1148,0.611498,6,4.48,중위험
6,동탄1동,4709,0.487365,1,4.39,중위험
7,장안면,812,0.655172,5,4.32,중위험
8,팔탄면,1764,0.581633,4,4.00,중위험
9,양감면,525,0.636190,5,3.90,중위험


In [61]:
# 저장
risk_df.to_csv(
    '../data/전처리_기존/행정동별위험도점수.csv',
    index=False, encoding='utf-8-sig'
)
print('저장 완료: 행정동별위험도점수.csv')
print()
print('=== 위험 등급 분포 ===')
print(risk_df['위험등급'].value_counts().to_string())
print()
print('=== 고위험 지역 ===')
display(risk_df[risk_df['위험등급'] == '고위험'][['행정동명','위험도점수','총상가수','주차수요비율']])

저장 완료: 행정동별위험도점수.csv

=== 위험 등급 분포 ===
위험등급
저위험    14
중위험    13
고위험     2

=== 고위험 지역 ===


,행정동명,위험도점수,총상가수,주차수요비율
0,우정읍,6.16,1372,0.628280
1,향남읍,6.15,5389,0.574132
